# Analisis dan Pemodelan Machine Learning — Paddy Dataset

Notebook ini menyusun alur lengkap dari **load data, pemeriksaan data, preprocessing, outlier, missing values, EDA, hingga pemodelan regresi** untuk memprediksi `Paddy yield(in Kg)`.

**Sebelum mulai**
- Letakkan file `paddy+dataset.zip` (atau file `paddydataset.csv` hasil ekstraksi) di folder yang sama dengan notebook.
- Jalankan sel secara berurutan dari atas ke bawah.
- Model ini memprediksi hasil padi dalam kilogram. Hasil evaluasi adalah estimasi pada data uji, bukan jaminan hasil panen di lapangan.
- Dataset yang disediakan berisi data pertanian; nilai ekstrem tidak otomatis salah. Karena itu, outlier akan diperiksa dan dilaporkan, bukan langsung dihapus.

## 1. Import library

**Sebelum dijalankan:** kita perlu pustaka untuk membaca dan mengolah data, visualisasi, serta membangun dan mengevaluasi model.

**Sesudah dijalankan:** jika tidak ada error, seluruh pustaka siap digunakan. Jika ada `ModuleNotFoundError`, instal paket yang disebutkan menggunakan `%pip install nama_paket`.

In [ ]:
import os
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

print("Library berhasil diimpor.")

## 2. Load dataset

**Sebelum dijalankan:** kode mencari `paddydataset.csv` atau `paddy+dataset.zip` di folder kerja notebook. Jika CSV berada di dalam ZIP, kode akan membacanya langsung tanpa perlu mengekstrak manual.

**Sesudah dijalankan:** `df` berisi tabel data mentah. Periksa pesan nama file dan ukuran data. Jika file tidak ditemukan, pindahkan ZIP/CSV ke folder notebook atau ubah `DATA_PATH`.

In [ ]:
DATA_PATH = Path(".")
csv_candidates = list(DATA_PATH.glob("*.csv"))
zip_candidates = list(DATA_PATH.glob("*.zip"))

if csv_candidates:
    data_file = csv_candidates[0]
    df = pd.read_csv(data_file)
    print(f"CSV dimuat: {data_file}")
elif zip_candidates:
    # Cari file ZIP yang berisi CSV
    chosen_zip = None
    csv_member = None
    for zip_path in zip_candidates:
        with zipfile.ZipFile(zip_path) as zf:
            members = [name for name in zf.namelist() if name.lower().endswith(".csv")]
            if members:
                chosen_zip, csv_member = zip_path, members[0]
                with zf.open(csv_member) as f:
                    df = pd.read_csv(f)
                break
    if chosen_zip is None:
        raise FileNotFoundError("ZIP ditemukan, tetapi tidak berisi file CSV.")
    print(f"CSV dimuat dari: {chosen_zip} → {csv_member}")
else:
    raise FileNotFoundError(
        "Tidak menemukan CSV/ZIP. Letakkan paddydataset.csv atau paddy+dataset.zip "
        "di folder yang sama dengan notebook."
    )

print("Ukuran data (baris, kolom):", df.shape)
display(df.head())

**Interpretasi setelah load:** setiap baris adalah satu observasi pada dataset. Kolom `Paddy yield(in Kg)` dipilih sebagai target yang ingin diprediksi; kolom lainnya menjadi kandidat fitur.

## 3. Memahami struktur data

**Sebelum dijalankan:** kita belum mengetahui tipe data, jumlah baris/kolom, maupun contoh nilai setiap kolom.

**Sesudah dijalankan:** `info()` menunjukkan tipe data dan kelengkapan kolom; `describe()` merangkum statistik numerik. Gunakan hasil ini untuk menemukan tipe data yang tidak sesuai dan rentang nilai yang perlu diperiksa.

In [ ]:
print("=== INFO DATA ===")
df.info()

print("\n=== 5 BARIS PERTAMA ===")
display(df.head())

print("\n=== STATISTIK NUMERIK ===")
display(df.describe().T)

print("\n=== JUMLAH NILAI UNIK ===")
display(df.nunique().sort_values().to_frame("jumlah_nilai_unik"))

## 4. Pemeriksaan kualitas data

**Sebelum dijalankan:** duplikasi, nilai kosong, dan kolom konstan belum diperiksa.

**Sesudah dijalankan:** tabel menampilkan jumlah missing values, duplikasi baris, dan kolom yang hanya memiliki satu nilai unik. Kolom konstan biasanya tidak membantu model; jangan menghapus kolom sebelum memahami maknanya.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print("=== MISSING VALUES PER KOLOM ===")
display(missing[missing > 0].to_frame("jumlah_missing") if (missing > 0).any()
        else pd.DataFrame({"status": ["Tidak ada missing value pada data mentah"]}))

print("Jumlah baris duplikat:", df.duplicated().sum())
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("Kolom konstan:", constant_cols if constant_cols else "Tidak ada")

## 5. Pembersihan dasar

**Sebelum dijalankan:** duplikasi persis dapat membuat observasi yang sama terhitung berulang. Nilai kosong perlu ditangani, tetapi pengisian nilai sebaiknya dipelajari dari data latih agar tidak terjadi kebocoran informasi.

**Sesudah dijalankan:** baris duplikat persis dihapus. Nilai kosong yang tersisa tidak diisi secara manual di sini; imputasi akan dilakukan di pipeline setelah pembagian data train/test.

In [ ]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Baris sebelum: {n_before:,}")
print(f"Baris setelah menghapus duplikat: {len(df):,}")
print(f"Duplikat yang dihapus: {n_before - len(df):,}")

## 6. Analisis outlier dengan metode IQR

**Sebelum dijalankan:** nilai ekstrem belum ditandai. Metode IQR menandai nilai di bawah `Q1 − 1.5×IQR` atau di atas `Q3 + 1.5×IQR`.

**Sesudah dijalankan:** tabel memberi jumlah dan persentase nilai yang terindikasi outlier untuk tiap kolom numerik. Ini hanya penanda statistik, bukan bukti kesalahan. Pada data pertanian, nilai tinggi/rendah bisa valid, jadi kita tidak menghapusnya secara otomatis.

In [ ]:
numeric_all = df.select_dtypes(include=np.number)
outlier_rows = []
for col in numeric_all.columns:
    q1, q3 = numeric_all[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        count = 0
    else:
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = int(((numeric_all[col] < lower) | (numeric_all[col] > upper)).sum())
    outlier_rows.append({
        "kolom": col, "jumlah_outlier_IQR": count,
        "persentase_%": round(count / len(df) * 100, 2)
    })
outlier_report = pd.DataFrame(outlier_rows).sort_values("jumlah_outlier_IQR", ascending=False)
display(outlier_report.head(20))

## 7. Exploratory Data Analysis (EDA)

**Sebelum dijalankan:** pola distribusi, perbedaan hasil panen, dan hubungan antarnilai belum terlihat.

**Sesudah dijalankan:** histogram memperlihatkan distribusi target; boxplot membantu melihat rentang dan nilai ekstrem; grafik kategori memperlihatkan rata-rata hasil panen menurut varietas dan jenis tanah. Grafik korelasi hanya memakai kolom numerik.

In [ ]:
target = "Paddy yield(in Kg)"
if target not in df.columns:
    raise KeyError(f"Kolom target '{target}' tidak ditemukan. Periksa nama kolom.")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[target], kde=True, ax=axes[0])
axes[0].set_title("Distribusi hasil panen padi")
axes[0].set_xlabel("Paddy yield (Kg)")
sns.boxplot(x=df[target], ax=axes[1])
axes[1].set_title("Boxplot hasil panen")
plt.tight_layout()
plt.show()

for col in ["Variety", "Soil Types"]:
    if col in df.columns:
        plt.figure(figsize=(10, 4))
        order = df.groupby(col)[target].mean().sort_values(ascending=False).index
        sns.barplot(data=df, x=col, y=target, order=order, errorbar=None)
        plt.title(f"Rata-rata hasil panen berdasarkan {col}")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Korelasi antarfitur numerik")
plt.tight_layout()
plt.show()

print("Ringkasan target:")
display(df[target].describe().to_frame("Paddy yield (Kg)"))

## 8. Menyiapkan fitur dan target

**Sebelum dijalankan:** model belum mengetahui kolom mana yang diprediksi dan kolom mana yang digunakan sebagai input.

**Sesudah dijalankan:** `X` berisi semua kolom selain target, sedangkan `y` berisi hasil panen. Data dibagi menjadi 80% data latih dan 20% data uji. Pembagian dilakukan sebelum imputasi dan encoding agar informasi dari data uji tidak bocor ke proses pelatihan.

In [ ]:
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()

print("Jumlah fitur:", X.shape[1])
print("Fitur numerik:", len(numeric_cols))
print("Fitur kategorikal:", len(categorical_cols))
print(f"Data latih: {X_train.shape[0]} baris")
print(f"Data uji:   {X_test.shape[0]} baris")

## 9. Preprocessing dan pemodelan

**Sebelum dijalankan:** sebagian model tidak dapat menerima teks kategori atau nilai kosong. Karena itu, pipeline akan:
1. Mengisi nilai numerik kosong dengan median.
2. Mengisi kategori kosong dengan modus.
3. Melakukan standardisasi fitur numerik.
4. Mengubah kategori menjadi kolom biner menggunakan One-Hot Encoding.
5. Melatih Random Forest Regressor.

**Sesudah dijalankan:** model terlatih dan siap memprediksi data uji. Pipeline memastikan langkah preprocessing yang sama diterapkan pada data latih maupun data uji. Random Forest dipakai sebagai baseline yang mampu menangkap hubungan nonlinier.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Pelatihan selesai. Prediksi data uji berhasil dibuat.")

## 10. Evaluasi model

**Sebelum dijalankan:** kita belum tahu seberapa dekat prediksi model dengan hasil panen aktual.

**Sesudah dijalankan:** metrik berikut mengukur performa pada data uji yang tidak digunakan untuk melatih model:
- **MAE:** rata-rata kesalahan absolut, dalam kilogram; makin kecil makin baik.
- **RMSE:** memberi penalti lebih besar pada kesalahan besar, dalam kilogram; makin kecil makin baik.
- **R²:** proporsi variasi target yang dijelaskan model; makin mendekati 1 umumnya makin baik. Nilai negatif berarti model lebih buruk daripada prediksi sederhana berupa rata-rata target pada data uji.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metrik": ["MAE (Kg)", "RMSE (Kg)", "R²"],
    "Nilai": [mae, rmse, r2]
})
display(metrics)

## 11. Membandingkan hasil aktual dan prediksi

**Sebelum dijalankan:** angka metrik belum menunjukkan contoh kesalahan pada tiap observasi.

**Sesudah dijalankan:** tabel membandingkan hasil aktual, hasil prediksi, dan selisihnya. Grafik aktual vs prediksi yang mendekati garis diagonal menunjukkan prediksi yang lebih dekat dengan nilai aktual.

In [ ]:
comparison = pd.DataFrame({
    "Aktual (Kg)": y_test.to_numpy(),
    "Prediksi (Kg)": y_pred,
})
comparison["Error absolut (Kg)"] = (comparison["Aktual (Kg)"] - comparison["Prediksi (Kg)"]).abs()
display(comparison.head(15))

plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Hasil aktual (Kg)")
plt.ylabel("Hasil prediksi (Kg)")
plt.title("Aktual vs Prediksi")
plt.tight_layout()
plt.show()

## 12. Fitur penting menurut Random Forest

**Sebelum dijalankan:** belum terlihat fitur yang paling banyak digunakan model untuk membuat prediksi.

**Sesudah dijalankan:** grafik menampilkan fitur dengan importance tertinggi setelah preprocessing. Untuk fitur kategori, nama fitur berasal dari hasil One-Hot Encoding. Importance menunjukkan kontribusi relatif dalam model ini, **bukan bukti sebab-akibat**.

In [ ]:
fitted_preprocessor = model.named_steps["preprocessing"]
feature_names = fitted_preprocessor.get_feature_names_out()
importances = model.named_steps["regressor"].feature_importances_

importance_df = pd.DataFrame({
    "fitur": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(importance_df.head(20))
plt.figure(figsize=(10, 7))
top_features = importance_df.head(15).sort_values("importance")
sns.barplot(data=top_features, x="importance", y="fitur", errorbar=None)
plt.title("15 fitur terpenting menurut Random Forest")
plt.tight_layout()
plt.show()

## 13. Kesimpulan dan langkah lanjutan

**Sesudah seluruh notebook dijalankan**, gunakan metrik evaluasi untuk menilai baseline. Jangan hanya melihat R²; perhatikan MAE/RMSE dalam satuan kilogram dan bandingkan dengan model sederhana atau algoritma lain.

Hal yang perlu diperhatikan:
- Dataset ini tidak memiliki missing values pada pemeriksaan awal (jika hasil Anda sama), tetapi imputasi tetap disiapkan agar pipeline lebih tahan terhadap data baru.
- Outlier IQR tidak dihapus otomatis karena nilai ekstrem bisa merupakan kondisi pertanian yang nyata.
- Pastikan fitur yang dipakai memang tersedia pada saat prediksi dilakukan. Fitur yang baru diketahui setelah panen dapat menyebabkan *data leakage* untuk penggunaan prediksi sebelum panen.
- Uji model lain (misalnya Gradient Boosting), lakukan cross-validation, dan evaluasi pada periode/lokasi berbeda sebelum dipakai untuk keputusan pertanian.
- Hasil ini menunjukkan pola prediksi dari dataset, bukan rekomendasi agronomi atau jaminan hasil panen.